# 04 - PySpark Concepts, End to End

A hands-on tour of the PySpark concepts that come up most often in data engineering
interviews. Every section has four parts:

1. **Concept** - the mental model, in plain language.
2. **Guarded code** - runnable PySpark that only executes when Spark is really available.
3. **Expected output** - what you should see, so the notebook still teaches without Spark.
4. **Interview note** - the trade-off a strong candidate mentions out loud.

## Prerequisites

- Notebook tooling: `uv sync --extra notebooks`
- Spark cells: `uv sync --extra notebooks --extra notebooks-spark`
- A **Java 17+** runtime on `PATH` (`java -version` must succeed). PySpark talks to the
  JVM; without Java, `import pyspark` can succeed while every job fails.

## How to read this notebook without Spark

This machine may not have PySpark or Java installed. That is fine. The first cell probes
the environment and sets `SPARK_AVAILABLE`. When it is `False`:

- every Spark cell prints `SKIPPED - install PySpark + Java 17 and re-run this cell`;
- the markdown cell *above* each Spark cell contains the expected output and the
  reasoning, so the concept is still fully covered;
- all `assert` statements live **inside** the Spark-available branch, so nothing fails
  just because Spark is missing.

Install Spark later, then use **Run All** (or
`python3 scripts/run_notebooks.py notebooks/04_pyspark_concepts.ipynb`) to materialise
the real outputs.

## Concepts covered

SparkSession and local mode; DataFrame creation with an explicit schema; lazy evaluation;
`select`/`filter`/`withColumn`/`rename`/`cast`; null handling; aggregations
(`groupBy`/`agg`/`pivot`); joins and broadcast hints; window functions; UDFs vs built-ins;
arrays and `explode`; partitioning and Parquet partition pruning; caching; `explain()`
plan reading; Adaptive Query Execution and skew; small files and compaction; a Structured
Streaming sketch; Delta and Iceberg notes on Databricks.

## Running this notebook in Google Colab

The **Open in Colab** badge in `notebooks/README.md` opens this file straight from GitHub. Colab provides a Java
runtime but not PySpark, so run one setup cell first:

```python
!pip install pyspark
```

Then choose **Runtime -> Restart session** and run all cells. The bootstrap cell below re-checks the environment and
turns the Spark cells on. If PySpark is unavailable the notebook still runs: every Spark cell prints a SKIPPED
message and the markdown above it explains the expected result.


## Environment bootstrap

**Expected output (no Spark here):**

```
PySpark installed : False
Java on PATH      : True
Java runnable     : False
Spark available   : False

This notebook will run in SKIPPED mode: ...
  - missing: a working Java 17+ runtime (the `java` on PATH cannot start)
  - missing: pyspark (the Python package)

To enable the Spark cells:
  1. uv sync --extra notebooks --extra notebooks-spark
  2. Install a Java 17+ runtime ...
  3. Re-run this cell.
```

**Interview note:** "it imports fine" is not "it runs". A quick capability probe up front
saves an hour of debugging a `JAVA_HOME`/`SparkContext` stack trace later - the same habit
you want in a production job that must fail fast on a broken runtime.

In [1]:
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except (ImportError, ValueError):  # `google` is not installed outside Colab
    IS_COLAB = False

# `shutil.which` only proves the binary exists. On some machines (e.g. macOS with only a
# stub at /usr/bin/java) the executable is present but cannot start a JVM, so probe it.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(
            ["java", "-version"], capture_output=True, text=True, timeout=60
        )
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
print(f"Colab runtime     : {IS_COLAB}")
print()

if SPARK_AVAILABLE:
    import pyspark
    print(f"PySpark version   : {pyspark.__version__}")
    print("Spark cells below will execute for real.")
else:
    print("This notebook will run in SKIPPED mode: every Spark cell prints a")
    print("SKIPPED message, and the markdown above each cell still teaches the concept.")
    print()
    if IS_COLAB:
        print("Colab detected. To enable the Spark cells here:")
        print("  1. Add a cell containing:  !pip install pyspark")
        print("  2. Run it, then choose Runtime -> Restart session")
        print("  3. Run all cells again (Colab already provides Java).")
    else:
        print("To enable the Spark cells:")
        print("  1. uv sync --extra notebooks --extra notebooks-spark")
        print("  2. Install a Java 17+ runtime (Temurin, or `brew install openjdk@17`)")
        print("  3. Make sure `java -version` works, then re-run this cell.")
    if not HAS_PYSPARK:
        print("  - missing: pyspark (the Python package)")
    if not HAS_JAVA:
        print("  - missing: a `java` executable on PATH")
    elif not JAVA_RUNS:
        print("  - missing: a working Java 17+ runtime (the `java` on PATH cannot start)")

PySpark installed : False
Java on PATH      : True
Java runnable     : False
Spark available   : False
Colab runtime     : False

This notebook will run in SKIPPED mode: every Spark cell prints a
SKIPPED message, and the markdown above each cell still teaches the concept.

To enable the Spark cells:
  1. uv sync --extra notebooks --extra notebooks-spark
  2. Install a Java 17+ runtime (Temurin, or `brew install openjdk@17`)
  3. Make sure `java -version` works, then re-run this cell.
  - missing: pyspark (the Python package)
  - missing: a working Java 17+ runtime (the `java` on PATH cannot start)


### 1. SparkSession and local mode

**Concept.** `SparkSession` is the single entry point to Spark SQL/DataFrames. It wraps a
`SparkContext` (the low-level RDD scheduler) plus the SQL catalog and configuration.
`master("local[*]")` runs the whole cluster - driver, one executor - inside this JVM, using
one core per CPU for parallelism. In a cluster you drop `.master(...)` and let the cluster
manager (YARN, Kubernetes, Databricks) supply it; hard-coding a master is the classic bug
that makes a job run on one node in production.

Key configs used here:

- `spark.sql.shuffle.partitions` - partitions produced by a shuffle (default 200).
- `spark.ui.enabled=false` - skip the web UI in a notebook.

**Expected output (with Spark):**

```
Spark version : 3.5.x
Master        : local[*]
App name      : de-interview-pyspark-concepts
self-check 1 passed: local session, correct app name
```

**Interview note:** describe the driver/executor split here. The driver plans work and
collects results; executors run tasks and hold cached blocks. `local[*]` collapses both
into one process, which is why it is a dev tool, not a deployment topology.

In [2]:
if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder
        .appName("de-interview-pyspark-concepts")
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")

    print("Spark version :", spark.version)
    print("Master        :", spark.sparkContext.master)
    print("App name      :", spark.sparkContext.appName)

    # --- self-check 1 ---------------------------------------------------
    assert spark.sparkContext.master.startswith("local"), spark.sparkContext.master
    assert spark.sparkContext.appName == "de-interview-pyspark-concepts"
    print("self-check 1 passed: local session, correct app name")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 2. DataFrames from Python data with an explicit schema

**Concept.** `spark.createDataFrame(data, schema)` turns local Python data into a
distributed DataFrame. Passing an explicit `StructType` (rather than letting Spark infer
types) is the production habit: inference reads the data twice, guesses `string` for dates
and numbers, and silently changes nullability. Explicit schemas also document the contract
and fail fast when the upstream shape changes.

`LongType` / `DoubleType` / `StringType` map to `bigint` / `double` / `string` in the
DataFrame. The third field of `StructField` is `nullable`.

**Expected output (with Spark):**

```
root
 |-- order_id: long (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_ts: string (nullable = true)

+--------+-----------+------+---------+-------------------+
|order_id|customer_id|amount|status   |order_ts           |
+--------+-----------+------+---------+-------------------+
|1       |c1         |120.5 |paid     |2026-10-01 10:00:00|
|2       |c2         |80.0  |paid     |2026-10-01 11:30:00|
|3       |c3         |200.0 |shipped  |2026-10-02 09:15:00|
|4       |c1         |15.75 |cancelled|2026-10-02 12:00:00|
|5       |c4         |60.0  |paid     |2026-10-03 08:45:00|
|6       |c2         |30.0  |returned |2026-10-03 16:20:00|
+--------+-----------+------+---------+-------------------+

self-check 2 passed: 6 rows, amount typed double, order_id non-nullable
```

**Interview note:** mention schema enforcement and corrupt-record handling together:
`mode=PERMISSIVE/FAILFAST/DROPMALFORMED` on the reader, and why `FAILFAST` in a curated
layer is usually right. A schema inferred in dev and drifted in prod is a top cause of
silent data corruption.

In [3]:
if SPARK_AVAILABLE:
    from pyspark.sql.types import (
        DoubleType,
        LongType,
        StringType,
        StructField,
        StructType,
    )

    orders_schema = StructType([
        StructField("order_id", LongType(), False),
        StructField("customer_id", StringType(), True),
        StructField("amount", DoubleType(), True),
        StructField("status", StringType(), True),
        StructField("order_ts", StringType(), True),
    ])

    orders_data = [
        (1, "c1", 120.50, "paid", "2026-10-01 10:00:00"),
        (2, "c2", 80.00, "paid", "2026-10-01 11:30:00"),
        (3, "c3", 200.00, "shipped", "2026-10-02 09:15:00"),
        (4, "c1", 15.75, "cancelled", "2026-10-02 12:00:00"),
        (5, "c4", 60.00, "paid", "2026-10-03 08:45:00"),
        (6, "c2", 30.00, "returned", "2026-10-03 16:20:00"),
    ]

    orders = spark.createDataFrame(orders_data, schema=orders_schema)
    orders.printSchema()
    orders.show(truncate=False)

    # --- self-check 2 ---------------------------------------------------
    assert orders.count() == 6
    assert orders.schema["amount"].dataType == DoubleType()
    assert orders.schema["order_id"].nullable is False
    print("self-check 2 passed: 6 rows, amount typed double, order_id non-nullable")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 3. Lazy evaluation: transformations vs actions

**Concept.** A **transformation** (`filter`, `withColumn`, `select`, `join`, `groupBy`)
only *describes* new work and returns a new immutable DataFrame. Nothing is scheduled. An
**action** (`count`, `show`, `collect`, `write`, `take`, `foreach`) asks for a concrete
result; only then does Catalyst optimise the whole chain into stages and tasks and the
scheduler launch them.

This is why a notebook cell that builds five DataFrames finishes instantly, and the cell
with a single `.count()` is the one that costs money. It is also why `withColumn` chains
compile into one scan rather than five passes.

**Expected output (with Spark):**

```
After two transformations, no job has been submitted yet.
isStreaming = False
(repeated)
paid rows: 3
```

The `explain()` call shows the plan for the full chain - proof the transformations were
accumulated, not executed one by one.

**Interview note:** the follow-up is always "so when does a job run?". Answer: at the
action, and one action = one job = one fresh pass over the source unless you cached or
reused a shuffled/saved result.

In [4]:
if SPARK_AVAILABLE:
    from pyspark.sql import functions as F

    # Transformation 1 - lazy.
    paid_only = orders.filter(F.col("status") == "paid")
    # Transformation 2 - still lazy, still nothing executed.
    with_tax = paid_only.withColumn("amount_with_tax", F.round(F.col("amount") * 1.08, 2))

    print("After two transformations, no job has been submitted yet.")
    print("isStreaming =", with_tax.isStreaming)
    with_tax.explain(True)  # prints the optimised plan; no data is read

    paid_rows = with_tax.count()  # <-- action: this is what triggers a job
    print("paid rows:", paid_rows)
    assert paid_rows == 3
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 4. `select`, `filter`, `withColumn`, `rename`, `cast`

**Concept.** These are the everyday column transformations and all are **narrow**
(no data exchange between executors):

- `select` / `drop` choose columns - push the projection down so Parquet reads fewer bytes.
- `filter` / `where` are aliases; predicates are pushed to the scan when possible.
- `withColumn` replaces or adds a column from an expression.
- `withColumnRenamed` renames; `alias` does it inline in a `select`.
- `cast` changes the type; do it explicitly and early, and know that a bad cast yields
  `null` (masking errors) instead of raising.

**Expected output (with Spark):**

```
root
 |-- order_id: long (nullable = false)
 |-- cust_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- amount_cents: long (nullable = true)

5 rows, columns: ['order_id', 'cust_id', 'amount', 'order_ts', 'amount_cents']
amount_cents first row = 12050
```

**Interview note:** say "narrow transformation" and explain why it matters: narrow steps
stay inside a partition, so they need no shuffle. Wide steps (`groupBy`, `join`, `distinct`,
`repartition`, window with a different partitioning) force an exchange and dominate cost.

In [5]:
if SPARK_AVAILABLE:
    from pyspark.sql import functions as F

    transformed = (
        orders
        .select("order_id", "customer_id", "amount", "status", "order_ts")
        .filter(F.col("status").isin("paid", "shipped"))
        .withColumn("order_ts", F.to_timestamp("order_ts"))
        .withColumn("amount_cents", (F.col("amount") * 100).cast("long"))
        .withColumnRenamed("customer_id", "cust_id")
        .drop("status")
    )

    transformed.printSchema()
    rows = transformed.collect()
    print(len(rows), "rows, columns:", transformed.columns)
    print("amount_cents first row =", rows[0]["amount_cents"])

    assert transformed.count() == 5
    assert "cust_id" in transformed.columns and "customer_id" not in transformed.columns
    assert dict(transformed.dtypes)["amount_cents"] == "bigint"
    assert dict(transformed.dtypes)["order_ts"] == "timestamp"
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 5. Null handling

**Concept.** Spark SQL is null-aware: `null == null` is `null` (not true), so use `isNull` /
`isNotNull` / `eqNullSafe` (`<=>`) rather than `==`. The practical toolkit:

- `isNull()`, `isNotNull()` for predicates;
- `na.drop(subset=[...])` to remove rows, `na.fill({...})` to substitute values;
- `coalesce(a, b, lit(default))` to pick the first non-null expression;
- `when(...).otherwise(...)` for conditional logic (handles null explicitly);
- `nullif` to turn a sentinel value back into a null.

`F.col("x") == None` does **not** work - the comparison yields null and filters everything.

**Expected output (with Spark):**

```
raw rows: 4
rows with any of amount/status null: 3
after fill+drop: 3 rows, 0 null amounts, 0 null statuses
coalesce produced 2 'unknown' statuses
```

**Interview note:** tie nulls to the metric. `avg` ignores nulls (denominator excludes
them), `sum` of all-null is null while `sum` with 0s is 0, and `count(col)` skips nulls
while `count(*)` does not. Interviewers use exactly this to test whether you trust a
dashboard number.

In [6]:
if SPARK_AVAILABLE:
    from pyspark.sql import functions as F

    nulls = spark.createDataFrame(
        [
            (1, "c1", 120.5, None),
            (2, "c2", None, "paid"),
            (3, None, 50.0, "paid"),
            (4, "c4", 0.0, None),
        ],
        "order_id long, customer_id string, amount double, status string",
    )
    nulls.show()

    missing = nulls.filter(F.col("amount").isNull() | F.col("status").isNull())
    print("raw rows:", nulls.count())
    print("rows with any of amount/status null:", missing.count())

    filled = nulls.na.fill({"amount": 0.0, "status": "unknown"}).na.drop(subset=["customer_id"])
    filled.show()
    print(
        "after fill+drop:",
        filled.count(),
        "rows,",
        filled.filter(F.col("amount").isNull()).count(),
        "null amounts,",
        filled.filter(F.col("status").isNull()).count(),
        "null statuses",
    )

    coalesced = nulls.withColumn("status_clean", F.coalesce(F.col("status"), F.lit("unknown")))
    print("coalesce produced", coalesced.filter(F.col("status_clean") == "unknown").count(), "'unknown' statuses")

    assert missing.count() == 3
    assert filled.count() == 3
    assert filled.filter(F.col("amount").isNull()).count() == 0
    assert filled.filter(F.col("status").isNull()).count() == 0
    assert coalesced.filter(F.col("status_clean") == "unknown").count() == 2
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 6. Aggregations: `groupBy` / `agg` / `pivot`

**Concept.** `groupBy(...).agg(...)` is a **wide** transformation: rows with the same key
must land in the same partition, so Spark performs a shuffle (a hash exchange) unless the
data is already co-partitioned. Costs to name in an interview: shuffle write + read, disk
spill, and the `spark.sql.shuffle.partitions` fan-out.

`pivot` rotates a low-cardinality column into columns. It must first buffer the distinct
pivot values, so Spark asks you to supply them explicitly when you know them -
`.pivot("product", ["gadget", "widget"])` - which avoids an extra pass over the data.

**Expected output (with Spark):**

```
+------+-------+------+-------+
|region|revenue|orders|avg_qty|
+------+-------+------+-------+
|north |105.0  |2     |6.5    |
|south |125.0  |2     |6.0    |
+------+-------+------+-------+

+------+------+------+
|region|gadget|widget|
+------+------+------+
|north |30.0  |25.0  |
|south |70.0  |12.5  |
+------+------+------+
```

The pivot table is revenue by product: widget = 10*2.5 = 25 (north) and 5*2.5 = 12.5
(south); gadget = 3*10 = 30 (north) and 7*10 = 70 (south).

**Interview note:** follow up `groupBy` with partition sizing. Too few post-shuffle
partitions and one task becomes the bottleneck (skew); too many and you pay scheduler and
small-file overhead. AQE can coalesce them for you - see section 14.

In [7]:
if SPARK_AVAILABLE:
    from pyspark.sql import functions as F

    sales = spark.createDataFrame(
        [
            ("2026-10-01", "north", "widget", 10, 2.5),
            ("2026-10-01", "south", "widget", 5, 2.5),
            ("2026-10-02", "north", "gadget", 3, 10.0),
            ("2026-10-02", "south", "gadget", 7, 10.0),
        ],
        "sale_date string, region string, product string, qty int, unit_price double",
    )

    per_region = (
        sales
        .withColumn("revenue", F.col("qty") * F.col("unit_price"))
        .groupBy("region")
        .agg(
            F.sum("revenue").alias("revenue"),
            F.count("*").alias("orders"),
            F.avg("qty").alias("avg_qty"),
        )
    )
    per_region.show()

    pivoted = (
        sales
        .withColumn("revenue", F.col("qty") * F.col("unit_price"))
        .groupBy("region")
        .pivot("product", ["gadget", "widget"])
        .agg(F.sum("revenue"))
    )
    pivoted.show()

    assert per_region.count() == 2
    assert dict(per_region.dtypes)["revenue"] == "double"
    assert set(pivoted.columns) == {"region", "gadget", "widget"}
    assert pivoted.filter(F.col("region") == "south").collect()[0]["gadget"] == 70.0
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 7. Joins: inner, left, broadcast hints, and why shuffle happens

**Concept.** A join keys two datasets on a common column. `how=` selects the semantics:
`inner` keeps matches only, `left` keeps every left row and null-fills the right side,
`left_anti` keeps left rows with **no** match (great for orphan detection), `left_semi`
keeps left rows that match (great for existence checks).

**Why a shuffle?** In a standard sort-merge/hash join, matching keys must be in the same
partition, so Spark repartitions both sides by the join key: a full network exchange.

**Broadcast join.** If one side is small (default limit `spark.sql.autoBroadcastJoinThreshold`
= 10 MB, configurable), Spark ships that whole side to every executor and joins locally -
no shuffle of the big table. You force or forbid it with
`F.broadcast(small_df)` / `F.broadcast` removal, and the hint is ignored if the side exceeds
the threshold. `left_anti` also avoids the wide shuffle when it can be turned into a
broadcast anti-join.

**Expected output (with Spark):**

```
inner rows: 2   (o3 has customer c9, which is not in customers)
left rows : 3
left rows with no matching customer: 1
broadcast join rows: 3
```

**Interview note:** the broadcast limit is about memory, not just time. Every executor must
hold the small side; a 4 GB "small" dimension table in `local[*]` or on a small executor
causes OOM. Also mention that a **skewed** join key (one key = 40% of rows) defeats the
shuffle join regardless of sizing - the fix is salting or AQE skew join.

In [8]:
if SPARK_AVAILABLE:
    from pyspark.sql import functions as F
    from pyspark.sql.functions import broadcast

    customers = spark.createDataFrame(
        [("c1", "Ada"), ("c2", "Grace"), ("c3", "Linus")],
        "customer_id string, name string",
    )
    small_orders = spark.createDataFrame(
        [("o1", "c1", 10.0), ("o2", "c2", 20.0), ("o3", "c9", 5.0)],
        "order_id string, customer_id string, amount double",
    )

    inner = small_orders.join(customers, on="customer_id", how="inner")
    left = small_orders.join(customers, on="customer_id", how="left")

    print("inner rows:", inner.count(), "  (o3 has customer c9, which is not in customers)")
    print("left rows :", left.count())
    print("left rows with no matching customer:", left.filter(F.col("name").isNull()).count())

    with_hint = small_orders.join(broadcast(customers), on="customer_id", how="left")
    with_hint.explain()  # look for BroadcastHashJoin in the physical plan
    print("broadcast join rows:", with_hint.count())

    # left_anti finds orphans without materialising right-hand columns.
    orphans = small_orders.join(customers, on="customer_id", how="left_anti")
    orphans.show()

    assert inner.count() == 2
    assert left.count() == 3
    assert left.filter(F.col("name").isNull()).count() == 1
    assert with_hint.count() == 3
    assert orphans.collect()[0]["order_id"] == "o3"
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 8. Window functions

**Concept.** A window function computes over a set of rows related to the current row
**without collapsing them** the way `groupBy` does. You define a `Window`:

- `partitionBy(...)` - the groups (this is what forces a shuffle);
- `orderBy(...)` - the ordering inside each group;
- `rowsBetween(...)` / `rangeBetween(...)` - the frame for aggregations.

The four workhorses:

- `row_number()` - unique 1..n per partition; the standard **dedupe** idiom
  (`rn = 1` keeps exactly one row per key even for exact ties).
- `rank()` / `dense_rank()` - leave gaps / do not leave gaps on ties, so ties produce the
  same rank and more than one "rank 1".
- `sum(...).over(unboundedPreceding..currentRow)` - running total.
- `lag()` / `lead()` - previous / next value, e.g. time between events.

Also used here: a **left anti** `/` **left semi** `join` (previous section) as the set-based
alternative to `row_number`-based dedupe.

**Expected output (with Spark):**

```
row_number dedupe keeps: 3 rows (one per customer)
c2 winner amount: 99.0
rows with rank() == 1     : 4   (the tied pair shares rank 1)
c2 rank() values: [1, 1, 3]   dense_rank() values: [1, 1, 2]
running total for c1: 175.0
rows where lag() is null: 3   (one first row per customer)
```

Note the tie: `rank()` jumps to 3 after the tied pair, `dense_rank()` continues at 2.
That is the difference interviewers probe for.

**Interview note:** "dedupe with `row_number` over the natural key ordered by
`updated_at desc`, then `rn = 1`" is the canonical CDC/upsert answer. Mention the tie-break:
put a deterministic second sort key in the `orderBy`, or the "latest" row is
non-deterministic across runs.

In [9]:
if SPARK_AVAILABLE:
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    events = spark.createDataFrame(
        [
            ("c1", "2026-10-01", 100.0),
            ("c1", "2026-10-03", 50.0),
            ("c1", "2026-10-02", 25.0),
            ("c2", "2026-10-01", 10.0),
            ("c2", "2026-10-01", 99.0),
            ("c2", "2026-10-01", 99.0),  # exact duplicate -> a real tie
            ("c3", "2026-10-05", 7.0),
        ],
        "customer_id string, order_date string, amount double",
    )

    by_customer_latest = Window.partitionBy("customer_id").orderBy(
        F.col("order_date").desc(), F.col("amount").desc()
    )

    # --- self-check 3: dedupe semantic ---------------------------------
    deduped = (
        events
        .withColumn("rn", F.row_number().over(by_customer_latest))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )
    print("row_number dedupe keeps:", deduped.count(), "rows (one per customer)")
    c2_amount = deduped.filter(F.col("customer_id") == "c2").collect()[0]["amount"]
    print("c2 winner amount:", c2_amount)

    ranked = events.withColumn("rk", F.rank().over(by_customer_latest))
    dense = events.withColumn("drk", F.dense_rank().over(by_customer_latest))
    print("rows with rank() == 1     :", ranked.filter(F.col("rk") == 1).count(), "  (the tied pair shares rank 1)")
    c2_ranks = [r["rk"] for r in ranked.filter(F.col("customer_id") == "c2").orderBy("rk").collect()]
    c2_dense = [r["drk"] for r in dense.filter(F.col("customer_id") == "c2").orderBy("drk").collect()]
    print("c2 rank() values:", c2_ranks, "  dense_rank() values:", c2_dense)

    running_window = (
        Window.partitionBy("customer_id")
        .orderBy("order_date")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    running = events.withColumn("running_amount", F.sum("amount").over(running_window))
    c1_total = running.filter(F.col("customer_id") == "c1").agg(F.max("running_amount")).collect()[0][0]
    print("running total for c1:", c1_total)

    lagged = events.withColumn(
        "prev_amount",
        F.lag("amount").over(Window.partitionBy("customer_id").orderBy("order_date")),
    )
    print("rows where lag() is null:", lagged.filter(F.col("prev_amount").isNull()).count())

    assert deduped.count() == 3
    assert c2_amount == 99.0
    assert ranked.filter(F.col("rk") == 1).count() == 4
    assert c2_ranks == [1, 1, 3]
    assert c2_dense == [1, 1, 2]
    assert c1_total == 175.0
    assert lagged.filter(F.col("prev_amount").isNull()).count() == 3
    print("self-check 3 passed: dedupe, rank tie semantics, running total, lag")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 9. UDFs vs built-in functions

**Concept.** A Python **UDF** ships a serialised Python function to each executor task,
runs it **row by row**, and converts every value between the JVM and the Python worker.
That breaks Catalyst's ability to optimise the expression and adds per-row pickling
overhead - often 2-10x slower than an equivalent built-in, and it blocks
whole-stage code generation (Tungsten). Rules of thumb:

1. Prefer `pyspark.sql.functions` (`when`, `regexp_extract`, `to_date`, `array_*`, ...).
2. If you truly need custom logic, prefer a **pandas UDF** (`@pandas_udf`) so work is
   vectorised per batch, or `F.udf` only for small, non-hot paths.
3. Never do network/DB calls inside a UDF - that is per-row I/O on every executor.

The example below computes the same tier with both approaches. Both give identical results;
the built-in is the one you should ship.

**Expected output (with Spark):**

```
+-------+---------+
|amount |tier_udf |
+-------+---------+
|120.5  |high     |
|80.0   |mid      |
|200.0  |high     |
|15.75  |low      |
|60.0   |mid      |
|30.0   |low      |
+-------+---------+

UDF 'high' rows   : 2
Built-in 'high' rows: 2
```

**Interview note:** the expected answer is "UDFs are a code smell in hot paths". Add the
escape hatch: pandas UDFs for vectorisation and, on Databricks, a Scala/Java UDF or
`transform`/higher-order SQL functions for zero-copy JVM execution.

In [10]:
if SPARK_AVAILABLE:
    from pyspark.sql import functions as F
    from pyspark.sql.types import StringType

    def _tier(amount):
        if amount is None:
            return "unknown"
        if amount >= 100:
            return "high"
        if amount >= 50:
            return "mid"
        return "low"

    tier_udf = F.udf(_tier, StringType())

    with_udf = orders.withColumn("tier_udf", tier_udf(F.col("amount")))
    with_builtin = orders.withColumn(
        "tier_builtin",
        F.when(F.col("amount") >= 100, "high")
        .when(F.col("amount") >= 50, "mid")
        .otherwise("low"),
    )

    with_udf.select("amount", "tier_udf").show()
    print("UDF 'high' rows   :", with_udf.filter(F.col("tier_udf") == "high").count())
    print("Built-in 'high' rows:", with_builtin.filter(F.col("tier_builtin") == "high").count())

    # Same answer, but the built-in version can be optimised and code-generated.
    assert with_udf.filter(F.col("tier_udf") == "high").count() == 2
    assert with_builtin.filter(F.col("tier_builtin") == "high").count() == 2
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 10. Arrays and `explode`

**Concept.** Semi-structured values (JSON lists, multi-valued tags) are natural as
`array<T>` columns. The core operators:

- `explode(col)` - one output row per element, **dropping null/empty arrays**.
- `explode_outer(col)` - same, but keeps a row with `null` when the array is null/empty.
- `array(...)`, `array_contains`, `array_distinct`, `size`, `concat`, `collect_list`
  for building and querying arrays;
- `posexplode` when the index matters.

`collect_list`/`collect_set` in a `groupBy` are the reverse operation: rows back into an
array.

**Expected output (with Spark):**

```
explode rows      : 3   (u1 has 2 tags, u2 has 1, empty/null arrays vanish)
explode_outer rows: 5   (u3 and u4 appear with a null tag)
tables with tag 'new': 4
```

**Interview note:** `explode` is a common hidden row multiplier - it inflates row counts
and therefore file sizes and shuffle volume. Mention the `explode_outer` choice explicitly:
silently dropping rows with empty arrays is a classic correctness bug in event pipelines.

In [11]:
if SPARK_AVAILABLE:
    from pyspark.sql import functions as F

    users = spark.createDataFrame(
        [
            ("u1", ["spark", "sql"]),
            ("u2", ["python"]),
            ("u3", []),
            ("u4", None),
        ],
        "user_id string, tags array<string>",
    )
    users.show(truncate=False)

    exploded = users.select("user_id", F.explode("tags").alias("tag"))
    exploded_outer = users.select("user_id", F.explode_outer("tags").alias("tag"))
    print("explode rows      :", exploded.count(), "  (u1 has 2 tags, u2 has 1, empty/null arrays vanish)")
    print("explode_outer rows:", exploded_outer.count(), "  (u3 and u4 appear with a null tag)")

    augmented = users.withColumn(
        "tags",
        F.array_distinct(
            F.concat(F.coalesce(F.col("tags"), F.array().cast("array<string>")), F.array(F.lit("new")))
        ),
    )
    augmented.show(truncate=False)
    print("tables with tag 'new':", augmented.filter(F.array_contains(F.col("tags"), "new")).count())

    assert exploded.count() == 3
    assert exploded_outer.count() == 5
    assert augmented.filter(F.array_contains(F.col("tags"), "new")).count() == 4
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 11. Partitioning and Parquet write/read with partition pruning

**Concept.** `partitionBy` on a **low-cardinality** column writes the data into a directory
per value (`order_date=2026-10-01/part-....parquet`) and removes that column from the files
themselves. A later query that filters on the partition column can then read only the
matching directories - **partition pruning** - which is the single biggest I/O win in most
lakehouse jobs. Parquet adds column pruning and predicate pushdown inside each file via its
footer metadata, plus compression and encoding.

Choose partition keys with care: thousands of tiny `date=.../hour=.../user_id=...`
directories is "over-partitioning" and is worse than not partitioning at all. Target
roughly 100 MB-1 GB per partition file.

**Expected output (with Spark):**

```
root
 |-- order_id: long (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- order_date: date (nullable = true)

rows read back        : 6
rows for 2026-10-01   : 2
partition directories : ['order_date=2026-10-01', 'order_date=2026-10-02', 'order_date=2026-10-03']
```

**Interview note:** mention pruning **and** its limits: filtering on a non-partition column
still scans every file (it only skips row groups via pushdown). This is exactly why
low-cardinality partition keys plus Parquet stats are the standard lakehouse layout, and
why tools like Delta/Iceberg add file-level min/max statistics.

In [12]:
if SPARK_AVAILABLE:
    import glob
    import os
    import tempfile

    from pyspark.sql import functions as F

    base_dir = tempfile.mkdtemp(prefix="de-interview-parquet-")
    parquet_path = os.path.join(base_dir, "orders_parquet")

    # --- self-check 4: round-trip and partition pruning ------------------
    (
        orders
        .withColumn("order_date", F.to_date("order_ts"))
        .write
        .mode("overwrite")
        .partitionBy("order_date")
        .parquet(parquet_path)
    )

    read_back = spark.read.parquet(parquet_path)
    read_back.printSchema()
    print("rows read back        :", read_back.count())

    pruned = read_back.filter(F.col("order_date") == F.to_date(F.lit("2026-10-01")))
    print("rows for 2026-10-01   :", pruned.count())
    pruned.explain()  # the scan should list only the matching partition directory

    partition_dirs = sorted(
        os.path.basename(p) for p in glob.glob(os.path.join(parquet_path, "order_date=*"))
    )
    print("partition directories :", partition_dirs)

    assert read_back.count() == 6
    assert pruned.count() == 2
    assert partition_dirs == [
        "order_date=2026-10-01",
        "order_date=2026-10-02",
        "order_date=2026-10-03",
    ]
    print("self-check 4 passed: partitioned Parquet round-trip with pruning")

    shutil.rmtree(base_dir, ignore_errors=True)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 12. Caching / persist and when not to cache

**Concept.** `cache()` = `persist(StorageLevel.MEMORY_AND_DISK)`. It stores the computed
partitions so the **next** action reuses them instead of recomputing the whole lineage.
`MEMORY_ONLY` drops partitions it cannot hold and recomputes them; `MEMORY_AND_DISK` spills
to disk; `DISK_ONLY` trades speed for memory. Always `unpersist()` when done - cached blocks
occupy executor memory that shuffle and joins also need.

**When NOT to cache:**

- the DataFrame is used once (cache is pure overhead);
- the source is already small/cheap, or you are going to `write` it immediately anyway;
- you cached "just in case" and the job now has memory pressure (the classic OOM culprit
  in long notebooks).

Also remember caching is **lazy** - the first action materialises it. Iterative algorithms
(ML training loops) and DataFrames reused 3+ times are the real wins.

**Expected output (with Spark):**

```
is_cached after cache()      : True
rows (materialises cache)    : 3
rows from cache              : 3
is_cached after unpersist()  : False
persisted with MEMORY_AND_DISK: True
```

**Interview note:** contrast `cache` with `checkpoint`. Cache keeps the **lineage** and is
lost when the session dies; checkpoint truncates the lineage to a reliable store, which is
what you need for very long chains or exactly-once recovery.

In [13]:
if SPARK_AVAILABLE:
    from pyspark import StorageLevel
    from pyspark.sql import functions as F

    expensive = (
        orders
        .withColumn("order_date", F.to_date("order_ts"))
        .groupBy("order_date")
        .agg(F.sum("amount").alias("revenue"))
        .cache()
    )
    print("is_cached after cache()      :", expensive.is_cached)

    first = expensive.count()   # first action materialises the cached blocks
    second = expensive.count()  # served from memory/disk, no recompute
    print("rows (materialises cache)    :", first)
    print("rows from cache              :", second)

    expensive.unpersist()
    print("is_cached after unpersist()  :", expensive.is_cached)

    persisted = orders.persist(StorageLevel.MEMORY_AND_DISK)
    persisted.count()
    print("persisted with MEMORY_AND_DISK:", persisted.is_cached)
    persisted.unpersist()

    assert first == 3 and second == 3
    assert not expensive.is_cached
    assert not persisted.is_cached
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 13. Reading `explain()` plans

**Concept.** `df.explain()` prints the physical plan; `explain(True)` also prints parsed,
analysed and optimised logical plans. Read it bottom-up (leaves to root):

- **Scan** - `FileScan parquet ... PushedFilters: [...]`; what was read and which predicates
  were pushed down. `PartitionFilters` shows partition pruning.
- **Filter / Project** - narrow work that stayed in-partition.
- **Exchange** - **a shuffle**. Any `Exchange` in the plan is your cost indicator.
- **HashAggregate** - a partial aggregate before the exchange, then a final one after.
- **BroadcastHashJoin** vs **SortMergeJoin** - whether the small side was broadcast.
- **AdaptiveSparkPlan** - AQE is enabled and may re-optimise at runtime.

`spark.sql("EXPLAIN FORMATTED <query>")` returns the plan as a DataFrame, which is handy
for asserting on the plan in tests.

**Expected output (with Spark):**

```
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[status#... ], functions=[sum(amount#...)])
   +- Exchange hashpartitioning(status#..., 8)
      +- HashAggregate(keys=[status#...], functions=[partial_sum(amount#...)])
         +- FileScan parquet ...
```

**Interview note:** being able to point at the `Exchange` and say "that is the shuffle, and
here is why" is the strongest practical signal in a Spark interview. Bring a real example:
a `groupBy` on a high-cardinality key with two `Exchange` nodes and a post-shuffle skew.

In [14]:
if SPARK_AVAILABLE:
    orders.createOrReplaceTempView("orders")

    plan_rows = spark.sql(
        """
        EXPLAIN FORMATTED
        SELECT status, sum(amount) AS revenue
        FROM orders
        GROUP BY status
        """
    ).collect()
    plan_text = "\n".join(row[0] for row in plan_rows)
    print(plan_text)

    lowered = plan_text.lower()
    assert len(plan_text) > 0
    assert "aggregate" in lowered
    # The GROUP BY is a wide transformation, so a shuffle must appear in the plan.
    assert "exchange" in lowered, plan_text
    print("plan contains an Exchange (shuffle) as expected for GROUP BY")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 14. Adaptive Query Execution (AQE) and skew handling

**Concept.** Since Spark 3.0, AQE re-optimises the plan **at runtime** using statistics from
completed shuffle stages. The three big features:

1. **Coalesce post-shuffle partitions** -
   `spark.sql.adaptive.coalescePartitions.enabled=true` merges many small shuffle
   partitions into `advisoryPartitionSizeInBytes`-sized ones. This is the modern answer to
   "my job has 200 tiny tasks".
2. **Skew join handling** - `spark.sql.adaptive.skewJoin.enabled=true` detects
   disproportionately large partitions and splits them into sub-tasks (usually by salting
   internally), so one straggler task no longer defines the job duration.
3. **Convert sort-merge join to broadcast join** at runtime if a side turns out to be small.

Without AQE, skew is handled manually: **salt** the hot keys (append a random suffix, explode
the other side), then re-aggregate.

**Expected output (with Spark):**

```
AQE enabled               : true
coalescePartitions        : true
skewJoin enabled          : true
advisory partition size   : 64m
+---------+---+
|status   |n  |
+---------+---+
|paid     |3  |
|cancelled|1  |
|returned |1  |
|shipped  |1  |
+---------+---+
```

**Interview note:** say what AQE *cannot* fix - a single key holding most rows still ends up
in one partition; AQE splits it, but if the source read itself is skewed you may need to
rebalance the input (`repartition` by a salted key, or fix upstream). Also note AQE is
disabled for Structured Streaming in many versions, so streaming skew needs manual design.

In [15]:
if SPARK_AVAILABLE:
    from pyspark.sql import functions as F

    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
    spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")

    print("AQE enabled               :", spark.conf.get("spark.sql.adaptive.enabled"))
    print("coalescePartitions        :", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
    print("skewJoin enabled          :", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
    print("advisory partition size   :", spark.conf.get("spark.sql.adaptive.advisoryPartitionSizeInBytes"))

    skew_probe = spark.sql(
        "SELECT status, count(*) AS n FROM orders GROUP BY status ORDER BY n DESC, status"
    )
    skew_probe.show()
    skew_probe.explain()  # AdaptiveSparkPlan: re-planned after shuffle statistics arrive

    assert spark.conf.get("spark.sql.adaptive.enabled") == "true"
    assert spark.conf.get("spark.sql.adaptive.skewJoin.enabled") == "true"
    assert skew_probe.count() == 4
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 15. Small files and compaction

**Concept.** Each Spark task writes one file per partition. A job with 2,000 partitions
writes 2,000 files; streaming or hourly ingestion multiplies that. Small files hurt because:

- every file has footer/metadata overhead in the driver and in the scan plan;
- object stores (S3) list and open files with high per-request latency;
- query planning times out on millions of files.

**Fixes:** `coalesce(n)` (narrow, no shuffle - it just merges partitions, best when
reducing slightly) or `repartition(n)` (full shuffle, even sizes - best when the current
sizes are wildly uneven). Then rewrite the table. Dedicated compaction jobs (Databricks
`OPTIMIZE`, Iceberg `rewrite_data_files`, Hudi compaction) do the same thing periodically.

**Expected output (with Spark, illustrative):**

```
small files before compaction: 24
files after compaction       : 2
rows preserved               : 6
```

The exact file counts depend on `repartition(8)` output and Spark's task count, but the
point is stable: fewer, larger files with identical rows.

**Interview note:** mention the **read** side too - `spark.sql.files.maxPartitionBytes`
controls how much a single task reads; too many small files means one task per file and a
storm of tiny tasks. The usual production answer is a scheduled compaction job, not ad-hoc
`coalesce` in every consumer.

In [16]:
if SPARK_AVAILABLE:
    import os
    import tempfile

    from pyspark.sql import functions as F

    small_base = tempfile.mkdtemp(prefix="de-interview-smallfiles-")
    raw_dir = os.path.join(small_base, "raw")
    compact_path = os.path.join(small_base, "compacted")

    for day in ("2026-10-01", "2026-10-02", "2026-10-03"):
        (
            orders
            .filter(F.to_date("order_ts") == F.to_date(F.lit(day)))
            .repartition(8)  # deliberately inflate the task -> file count
            .write
            .mode("overwrite")
            .parquet(os.path.join(raw_dir, f"ds={day}"))
        )

    before = [
        os.path.join(root, f)
        for root, _, files in os.walk(raw_dir)
        for f in files
        if f.endswith(".parquet")
    ]
    print("small files before compaction:", len(before))

    # Write the compacted copy to a sibling directory so re-reading `raw_dir` after the
    # write cannot pick up the compacted files (which would double-count the rows).
    raw = spark.read.option("recursiveFileLookup", "true").parquet(raw_dir)
    compacted = raw.coalesce(2)  # merge partitions without a shuffle
    compacted.write.mode("overwrite").parquet(compact_path)

    after = [
        os.path.join(root, f)
        for root, _, files in os.walk(compact_path)
        for f in files
        if f.endswith(".parquet")
    ]
    print("files after compaction       :", len(after))
    print("rows preserved               :", compacted.count())

    assert compacted.count() == 6
    assert 0 < len(after) < len(before)
    shutil.rmtree(small_base, ignore_errors=True)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### 16. Structured Streaming sketch (code shown, never executed)

Structured Streaming treats a stream as an unbounded table you keep querying. Three things
an interviewer listens for:

- **`readStream` / `writeStream`** - the same DataFrame API as batch, so joins and
  aggregations transfer directly.
- **Checkpointing** - `checkpointLocation` persists offsets and state so a restart resumes
  where it stopped instead of reprocessing or losing data.
- **Output mode / watermark / trigger** - `append` vs `update` vs `complete`, and a
  watermark to bound state for event-time aggregations.

The block below is intentionally **not executed** by this notebook (it needs a Kafka
broker and a table sink). Read it as a reference shape.

```python
# Reference only - NOT executed by scripts/run_notebooks.py.
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructType, TimestampType

order_schema = StructType() \
    .add("order_id", StringType()) \
    .add("customer_id", StringType()) \
    .add("amount", DoubleType()) \
    .add("status", StringType()) \
    .add("order_ts", TimestampType())

stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "latest")
    .load()
)

parsed = (
    stream
    .select(F.from_json(F.col("value").cast("string"), order_schema).alias("o"))
    .select("o.*")
    .withWatermark("order_ts", "10 minutes")
)

query = (
    parsed
    .groupBy(F.window("order_ts", "5 minutes"), "status")
    .agg(F.count("*").alias("n"), F.sum("amount").alias("revenue"))
    .writeStream
    .outputMode("update")
    .option("checkpointLocation", "s3://company-checkpoints/orders_5m/")
    .trigger(processingTime="1 minute")
    .toTable("main.gold.orders_5m")  # or .foreachBatch(...) / .format("delta").start(path)
)

query.awaitTermination()
```

**Interview note:** the killer follow-up is **exactly-once**. Structured Streaming gives
end-to-end exactly-once only when the source is replayable (Kafka with committed offsets),
the checkpoint is durable, and the sink is idempotent/transactional (Delta, Iceberg, or a
`foreachBatch` merge keyed on a deterministic batch id). `update` mode without a watermark
means unbounded state.

### 17. Delta Lake and Apache Iceberg notes (mostly a Databricks conversation)

Both projects add a **transaction log / metadata layer** on top of Parquet, which is what
turns a directory of files into a reliable table. What they share:

- **ACID transactions and atomic metadata commits** - a failed write does not leave a
  half-visible table.
- **Time travel and snapshots** - query `VERSION AS OF` / `TIMESTAMP AS OF`.
- **Schema evolution and enforcement** - `mergeSchema`, `ALTER TABLE ... ADD COLUMN`.
- **`MERGE` / upsert** - the standard CDC pattern: match on the primary key, update matched
  rows, insert new ones, delete flagged ones.
- **Small-file management** - Delta `OPTIMIZE` (with `ZORDER BY` for data skipping),
  Iceberg `rewrite_data_files` + `rewrite_manifests`.
- **Statistics for data skipping** - per-file min/max, so `WHERE` on non-partition columns
  can still prune files.

Differences worth calling out:

| | Delta Lake | Apache Iceberg |
|---|---|---|
| Metadata | JSON + Parquet transaction log (`_delta_log`) | Metadata JSON + manifest files |
| Format spec | Delta protocol, originally Databricks | Open table spec, vendor-neutral |
| Hidden partitioning | Partition columns are real columns | `PARTITION BY` transforms (`days(ts)`, `bucket(id, 16)`) hide the column |
| Pushdown | Data skipping + ZORDER | Partition transforms + sort orders + manifests |
| Typical home | Databricks, Spark, some Trino/Flink | Trino, Flink, Spark, Snowflake, BigQuery |

Databricks-specific habits: Unity Catalog for governance, `OPTIMIZE` + `ZORDER`,
`VACUUM` to expire old files, `MERGE INTO` for upserts, and `AUTO OPTIMIZE` /
`AUTO COMPACT` for streaming sinks. On Databricks the **Photon** engine and
**predictive optimization** do much of the tuning, but partitioning, skew, and file sizing
are still design decisions you own.

**Expected output:** markdown only - nothing to execute.

**Interview note:** if asked "Delta or Iceberg?", answer with constraints: what query engines
must read the table, whether you need vendor neutrality, whether hidden partitioning and
sort orders matter, and what the team already runs. Then talk about the cost of migration -
metadata is the easy part; downstream `MERGE` jobs and permission models are not.

## Interview questions

**1. What is a shuffle, and why is it expensive?**
A shuffle redistributes rows across partitions so that rows sharing a key (or a partition
requirement) land together. It is expensive because it writes intermediate data to executor
local disk, transfers it across the network, reads it back, and can spill. It also forces a
stage boundary, so the whole upstream stage must complete first. Wide transformations cause
shuffles: `groupBy`, `join`, `distinct`, `repartition`, `orderBy`, and window functions with
a partition key. Narrow transformations (`filter`, `select`, `withColumn`, `union` of
same-partitioning data) do not.

**2. Narrow vs wide transformations - give examples and consequences.**
Narrow: each output partition depends on exactly one input partition (`map`, `filter`,
`select`, `withColumn`, `union`). No data movement; pipelined inside a stage. Wide: output
partitions depend on many input partitions (`groupBy`, `join`, `distinct`, `repartition`,
`sort`). These create a new stage, a shuffle, and a barrier. Rule of thumb: every `Exchange`
node in `explain()` marks a wide operation.

**3. When does a broadcast join help, and what are its limits?**
It helps when one side is small enough to fit in executor memory: Spark ships that side to
every executor and avoids shuffling the large side, turning a sort-merge join into a
map-side broadcast join. Limits: the broadcast side must fit in memory on **every** executor
(default threshold `spark.sql.autoBroadcastJoinThreshold=10MB`, and you can hint with
`F.broadcast`); too large and you get OOM or a silently ignored hint; broadcast also does
not help if the *large* side is skewed; and broadcasts are re-sent per stage/executor, so a
"small" 500 MB table is not actually small.

**4. What is Adaptive Query Execution and what problem does it solve?**
AQE re-optimises a query at runtime using actual shuffle statistics, instead of relying only
on plan-time estimates (which are often wrong). It coalesces many small post-shuffle
partitions (solving the 200-partition default problem), splits skewed partitions for skewed
joins, and can switch a sort-merge join to broadcast when a side turns out to be small.
Enable with `spark.sql.adaptive.enabled=true` (default since Spark 3.2) plus
`coalescePartitions.enabled` and `skewJoin.enabled`.

**5. How do you handle data skew, and what can AQE not fix?**
Detect it: one task takes far longer, one partition's spill/bytes dwarfs the others, `explain`
shows an unbalanced exchange. Fixes, in order: let AQE's skew join split the hot partitions;
salting the hot key (append a random suffix 0..N to the key on both sides, aggregate, then
re-aggregate without the suffix); broadcast the small side if skew comes from the join;
`repartition` on a better key; or fix the root cause (a null/placeholder key such as `"-1"`
holding most rows). AQE cannot fix a **skewed source read** - if one input file or one Kafka
partition holds most of the data, you must rebalance at ingestion.

**6. How do you choose partition columns, and what is over-partitioning?**
Partition on a low-to-medium cardinality column that is almost always in the `WHERE` clause
(date, region), that produces roughly 100 MB-1 GB per partition, and whose values are evenly
distributed. Over-partitioning is millions of tiny directories (e.g. partitioning by
`user_id` or by date+hour+minute), which inflates metadata, listing latency, and task count,
and defeats the point. Alternative for high-cardinality queries: leave the table
unpartitioned/bucketed and rely on Parquet/Delta/Iceberg statistics for file skipping.

**7. When should you cache, and when should you not?**
Cache when a DataFrame or an expensive intermediate is reused by 2-3+ actions, or in an
iterative algorithm. Do not cache when it is used once, when the source is already a small
in-memory table, when you are about to write the result anyway, or when it would push the
job into memory pressure (spilled cache is often slower than recomputing a cheap lineage).
Caching is lazy - the first action materialises it - and you should `unpersist()` when done.

**8. UDFs: why are they slow, and what are the alternatives?**
A Python UDF serialises the function and every row between the JVM and a Python worker, runs
row by row, prevents Catalyst from optimising the expression, and blocks whole-stage code
generation. Alternatives: built-in `pyspark.sql.functions` (always first choice);
higher-order SQL functions (`transform`, `filter`, `aggregate`) on arrays/maps; pandas UDFs
(`@pandas_udf`) which vectorise per batch via Arrow; or a Scala/Java UDF for JVM-native
execution. Reserve plain Python UDFs for cold paths.

**9. What is checkpointing, and how does it differ from caching?**
Checkpointing writes the DataFrame to a reliable store (HDFS/S3) and **truncates the
lineage**, so recovery does not replay an ever-growing chain of transformations. Caching
keeps the lineage and lives only in executor memory/disk for the session. Use checkpoints
for very long lineage chains, iterative algorithms, and streaming (where
`checkpointLocation` stores offsets and state). Checkpoint files must be cleaned up
separately; they are not freed by `unpersist`.

**10. How does Structured Streaming achieve exactly-once?**
Exactly-once needs three things together: a **replayable source** with committed offsets
(Kafka, or a file source), a **durable checkpoint** recording the offsets and state for each
micro-batch, and an **idempotent/transactional sink** (Delta or Iceberg tables, or a
`foreachBatch` upsert keyed on the batch/version). The checkpoint lets a restart resume
without reprocessing; the transactional sink makes a partially written batch invisible or
safely overwritten, so at-least-once delivery becomes exactly-once state.

**11. Driver vs executor memory - what lives where, and what are the common OOMs?**
The driver runs the `SparkSession`, the DAG scheduler, the Catalyst planner, and collects
results. It owns memory for `collect()`, `toPandas()`, broadcast variables under the
broadcast threshold, and long-lived objects in notebook loops. Executors run tasks, hold
shuffle data, cached blocks, and broadcast joins. Common OOMs: `collect()`/`toPandas()` on a
large result (driver OOM); caching too much or broadcast joins that are not actually small
(executor OOM); `groupBy` on a skewed key with too few partitions (executor OOM/spill); and
Python UDF workers exhausting memory when each row builds large objects.

**12. How do you deal with small files?**
Prevent them at write time with sensible `repartition`/`coalesce` (one file per output
partition), and run scheduled compaction: Delta `OPTIMIZE`, Iceberg `rewrite_data_files`,
Hudi compaction, or a `coalesce`/`repartition` rewrite job. On the read side, bound
`spark.sql.files.maxPartitionBytes` and avoid one task per file. Small files hurt planning
time, object-store listing latency, and per-file metadata overhead, and they break
predicate pushdown efficiency because each file is opened individually.

**13. DataFrames vs RDDs - when would you still use an RDD?**
DataFrames go through Catalyst (rule-based + cost-based optimisation), Tungsten
(code generation, off-heap memory, compact binary layout), and a typed schema, so they are
almost always faster and safer. RDDs give you arbitrary Python/Scala objects and fine control
over partitioning, which is occasionally needed for custom iterative algorithms, untyped
binary processing, or porting legacy code. The interview answer: default to DataFrames,
reach for RDDs (`df.rdd`) only when the API genuinely cannot express the logic, and know
that leaving the DataFrame API loses all optimizer benefits.

**14. `repartition` vs `coalesce`?**
`repartition(n)` performs a **full shuffle** and produces `n` partitions of roughly even
size; use it to increase parallelism or to rebalance badly skewed data, optionally on a key
(`repartition(n, "key")`). `coalesce(n)` is narrow: it merges existing partitions without a
shuffle, so it can only decrease the count and the resulting partitions may be uneven; use
it before a write to reduce file count. Common pattern: `repartition` for balance, then
`coalesce`/`repartition` before writing to control file sizes.

**15. How do you debug a slow Spark job?**
Start from the Spark UI: find the longest stage, then the slowest task inside it, and compare
its input size and spill to its peers - a large gap means skew. Read `explain()` for
`Exchange` nodes, broadcast vs sort-merge joins, and full scans where you expected pruning.
Check partition counts (`spark.sql.shuffle.partitions`, input file counts), shuffle
spill-to-disk metrics, GC time, and whether a UDF or `collect()` is on the hot path. Then fix
the highest-leverage thing: broadcast/`repartition` the join, enable AQE, compact files,
remove the UDF, or add a `filter` earlier so less data is read.

## Running this notebook with Spark for real

```bash
# from the repo root
uv sync --extra notebooks --extra notebooks-spark
java -version                      # must be 17+
python3 scripts/run_notebooks.py notebooks/04_pyspark_concepts.ipynb
```

The runner executes every cell and saves the outputs back into the notebook. With PySpark
and Java present, `SPARK_AVAILABLE` becomes `True`, all SKIPPED messages turn into real
results, and the four self-check assertion blocks run for real.

**Expected summary with Spark:** `Spark available: True`, `SparkSession stopped.`
**Expected summary here:** `Spark available: False`, all Spark cells reported SKIPPED.

In [17]:
print("Spark available:", SPARK_AVAILABLE)
if SPARK_AVAILABLE:
    print("All Spark cells executed against a local SparkSession.")
    print("Re-run this notebook whenever you change the code to refresh the saved outputs.")
else:
    print("All Spark cells were skipped; every concept above still includes its expected output.")
    if not HAS_PYSPARK:
        print("Missing: pyspark  ->  uv sync --extra notebooks --extra notebooks-spark")
    if not (HAS_JAVA and JAVA_RUNS):
        print("Missing: a working Java 17+ runtime  ->  install Temurin or openjdk@17")

Spark available: False
All Spark cells were skipped; every concept above still includes its expected output.
Missing: pyspark  ->  uv sync --extra notebooks --extra notebooks-spark
Missing: a working Java 17+ runtime  ->  install Temurin or openjdk@17


In [18]:
if SPARK_AVAILABLE:
    spark.stop()
    print("SparkSession stopped.")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell
